# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

この実装はAB予測＋AM修正1回のPECE法で、陰的AM方程式を収束まで解く方式ではありません。完全な陰的法の安定性をそのまま適用しないでください。4次法は初期値に加え3ステップをRK4で準備します。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


In [ ]:
import numpy as np
import time 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
parameters = {
    'axes.labelsize': 26,           # Set the font size of the x-axis and y-axis labels to 26
    'xtick.labelsize': 24,          # Set the font size of the x-axis tick labels to 24
    'ytick.labelsize': 24,          # Set the font size of the y-axis tick labels to 24
    'axes.titlesize': 24,           # Set the font size of the axis titles (not the overall graph title) to 24
    'figure.titlesize': 24,         # Set the font size of the overall graph title to 24
    'legend.fontsize': 22,          # Set the font size of the legend to 22
    'figure.figsize': [10, 6],      # Set the default figure size (width 10 inches, height 6 inches)
    'axes.linewidth': 2.0           # Set the linewidth of the axes
}
plt.rcParams.update(parameters)

In [ ]:
def f(t, y, alpha):
    """
    Defines the right-hand side of the differential equation dy/dt = -alpha * y.

    Parameters:
        t (float): The current time
        y (float): The current value of y
        alpha (float): The coefficient alpha in the equation

    Returns:
        float: The value of dy/dt
    """
    return -alpha * y

def analytical_solution(t, alpha):
    """
    Computes the analytical solution y(t) = e^(-alpha * t).

    Parameters:
        t (float or np.ndarray): The time at which to evaluate the solution
        alpha (float): The coefficient alpha in the equation

    Returns:
        float or np.ndarray: The analytical solution y(t)
    """
    return np.exp(-alpha * t)

## Explanation of the `euler_method` Function

This function implements the **explicit Forward Euler method** for solving ordinary differential equations (ODEs) of the form \( \frac{dy}{dt} = f(t, y) \). The Forward Euler method is an **explicit** method, meaning it uses the current information at each time step to compute the next value directly.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.
   - The next value \( y_{\text{new}} \) is computed using the Forward Euler formula:
     \[
     y_{\text{new}} = y_n + h \cdot f(t_n, y_n, \alpha)
     \]
     This formula updates the solution at the next time step by using the current derivative information from \( f(t_n, y_n, \alpha) \).

3. **Update the Solution**:
   - Once \( y_{\text{new}} \) is computed, it is stored in `y_values[n+1]`.

4. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Time stepping**: The solution progresses step-by-step from \( t_0 \) to \( t_{\text{end}} \), updating the solution using the Forward Euler method formula.
2. **Explicit Update**: At each time step, the current solution \( y_n \) and the function value \( f(t_n, y_n, \alpha) \) are used to compute the next value \( y_{\text{new}} \).
3. **Simple and Direct**: The Forward Euler method is straightforward but can become unstable for stiff ODEs or when the step size \( h \) is too large.

### Key Advantages:
- **Simple and Easy to Implement**: The Forward Euler method is an explicit method, making it computationally cheap and easy to implement.
- **Direct Calculation**: The next solution value is calculated directly from the current information without needing to solve an implicit equation.
  
### Limitations:
- **Limited Stability**: The Forward Euler method may become unstable for large step sizes or for stiff ODEs, meaning that smaller step sizes are often required for accurate results.
- **First-Order Accuracy**: The method has first-order accuracy, meaning the error decreases linearly as the step size is reduced.

In [ ]:
# Define the Forward Euler Method Function
def euler_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the explicit Forward Euler method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]
        y_new = y_n + h * f(t_n, y_n, alpha)
        y_values[n + 1] = y_new

    return t_values, y_values

## Explanation of the `rk2_heun_method` Function

This function implements **Heun's method**, which is an improved version of the Euler method, also known as the **second-order Runge-Kutta (RK2)** method. It improves upon the basic Forward Euler method by using an additional step to correct the predicted value, resulting in greater accuracy. The method consists of two steps: a **predictor step** and a **corrector step**.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.

3. **Predictor Step**:
   - A **Forward Euler step** is first performed to calculate an initial guess for the next solution point:
     \[
     y_{\text{predictor}} = y_n + h \cdot f(t_n, y_n, \alpha)
     \]
   - This predicted value \( y_{\text{predictor}} \) serves as an estimate of the solution at \( t_{\text{new}} = t_n + h \).

4. **Corrector Step**:
   - The corrector step improves the solution by taking the **average slope** between the current slope and the slope at the predicted point:
     \[
     y_{\text{corrector}} = y_n + \frac{h}{2} \cdot \left( f(t_n, y_n, \alpha) + f(t_{\text{new}}, y_{\text{predictor}}, \alpha) \right)
     \]
   - This correction adjusts the predicted value using the average of the initial and predicted slopes.

5. **Update the Solution**:
   - The corrected value \( y_{\text{corrector}} \) is stored in `y_values[n+1]`.

6. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Predictor Step**: First, the method uses the Forward Euler step to predict the next value using the slope at the current point.
2. **Corrector Step**: Then, it corrects the predicted value by averaging the slopes from both the current point and the predicted point, increasing the accuracy.
3. **Improved Accuracy**: By including the corrector step, Heun’s method achieves second-order accuracy, offering a significant improvement over the basic Forward Euler method.

### Key Advantages:
- **Improved Accuracy**: Heun’s method is more accurate than the standard Euler method because it accounts for changes in the slope over the time step.
- **Second-Order Accuracy**: The method has second-order accuracy, meaning the error decreases quadratically as the step size \( h \) is reduced.
  
### Limitations:
- **More Computational Effort**: While more accurate, Heun’s method requires two evaluations of the function \( f \) per time step (for both the predictor and corrector), making it more computationally expensive than the Forward Euler method.

In [ ]:
def rk2_heun_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using Heun's method (improved Euler method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Forward Euler)
        y_predictor = y_n + h * f(t_n, y_n, alpha)

        # Corrector step (average slope)
        t_new = t_n + h
        y_corrector = y_n + (h / 2) * (f(t_n, y_n, alpha) + f(t_new, y_predictor, alpha))

        y_values[n + 1] = y_corrector

    return t_values, y_values

## Explanation of the `rk2_midpoint_method` Function

This function implements the **midpoint method**, which is a second-order **Runge-Kutta (RK2)** method for solving ordinary differential equations (ODEs) of the form \( \frac{dy}{dt} = f(t, y) \). The midpoint method improves upon the basic Euler method by using an intermediate value (midpoint) to compute a more accurate slope, providing better accuracy.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.

3. **Predictor Step** (Euler's step to midpoint):
   - The first slope \( k_1 \) is computed using the initial slope at \( t_n \) and \( y_n \):
     \[
     k_1 = f(t_n, y_n, \alpha)
     \]
   - This slope is used to predict the midpoint value \( y_{\text{mid}} \):
     \[
     y_{\text{mid}} = y_n + \frac{h}{2} \cdot k_1
     \]
   - The midpoint value gives an estimate of \( y \) at the midpoint of the interval.

4. **Corrector Step** (using slope at the midpoint):
   - The second slope \( k_2 \) is computed using the function evaluated at the midpoint \( t_{\text{mid}} = t_n + \frac{h}{2} \):
     \[
     k_2 = f\left( t_n + \frac{h}{2}, y_{\text{mid}}, \alpha \right)
     \]
   - The corrected value \( y_{\text{new}} \) is then calculated using this slope:
     \[
     y_{\text{new}} = y_n + h \cdot k_2
     \]

5. **Update the Solution**:
   - The corrected value \( y_{\text{new}} \) is stored in `y_values[n+1]`.

6. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Predictor Step (Midpoint)**: The midpoint method first predicts an intermediate value \( y_{\text{mid}} \) by taking an Euler step using the initial slope \( k_1 \).
2. **Corrector Step**: The slope at the midpoint \( k_2 \) is used to correct the solution, providing a more accurate estimate of the solution at the next time point.
3. **Improved Accuracy**: By using the slope at the midpoint rather than at the start of the step, the midpoint method achieves second-order accuracy.

### Key Advantages:
- **Second-Order Accuracy**: The midpoint method is a second-order Runge-Kutta method, offering better accuracy than the standard Euler method.
- **Use of Midpoint**: By evaluating the slope at the midpoint, this method provides a more accurate estimate of the next solution value.

### Limitations:
- **More Computational Effort**: Similar to other second-order methods, this method requires two function evaluations per time step, making it computationally more expensive than the Forward Euler method.

In [ ]:
def rk2_midpoint_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the RK2 method (midpoint method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Euler's step to midpoint)
        k1 = f(t_n, y_n, alpha)
        y_mid = y_n + (h / 2) * k1  # Midpoint value

        # Corrector step (use slope at the midpoint)
        k2 = f(t_n + h / 2, y_mid, alpha)
        y_new = y_n + h * k2  # Update using the slope at midpoint

        y_values[n + 1] = y_new

    return t_values, y_values

## Explanation of the `rk4_method` Function

This function implements the **4th-order Runge-Kutta (RK4) method**, which is a highly accurate method for solving ordinary differential equations (ODEs) of the form \( \frac{dy}{dt} = f(t, y) \). The RK4 method improves upon lower-order methods by using four function evaluations per step to achieve higher accuracy.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.

3. **RK4 Coefficients**:
   - The RK4 method calculates four intermediate slopes (coefficients) \( k_1, k_2, k_3, k_4 \) at different points within the time step:
     - \( k_1 \) is the slope at the beginning of the step:
       \[
       k_1 = f(t_n, y_n, \alpha)
       \]
     - \( k_2 \) is the slope at the midpoint, using the first slope \( k_1 \):
       \[
       k_2 = f(t_n + \frac{h}{2}, y_n + \frac{h}{2} \cdot k_1, \alpha)
       \]
     - \( k_3 \) is another midpoint slope, but based on \( k_2 \):
       \[
       k_3 = f(t_n + \frac{h}{2}, y_n + \frac{h}{2} \cdot k_2, \alpha)
       \]
     - \( k_4 \) is the slope at the end of the step:
       \[
       k_4 = f(t_n + h, y_n + h \cdot k_3, \alpha)
       \]

4. **Update the Solution**:
   - The next value \( y_{\text{new}} \) is computed using a weighted average of the four slopes:
     \[
     y_{\text{new}} = y_n + \frac{h}{6} \cdot (k_1 + 2 \cdot k_2 + 2 \cdot k_3 + k_4)
     \]
   - This formula provides a highly accurate estimate of the solution at the next time step.

5. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **RK4 Coefficients**: The method calculates four slopes, \( k_1, k_2, k_3, k_4 \), at different points within the time step, providing a better approximation than simpler methods.
2. **Weighted Average**: The final solution at each step is calculated using a weighted average of these slopes, ensuring higher accuracy.
3. **Fourth-Order Accuracy**: The RK4 method achieves fourth-order accuracy, making it one of the most commonly used methods for solving ODEs.

### Key Advantages:
- **High Accuracy**: The RK4 method has fourth-order accuracy, meaning the error decreases as \( O(h^4) \), where \( h \) is the step size. This makes it much more accurate than lower-order methods such as Euler or RK2 methods.
- **Wide Application**: RK4 is widely used in practice due to its balance between accuracy and computational cost.

### Limitations:
- **More Computational Effort**: RK4 requires four evaluations of the function \( f \) per time step, making it more computationally expensive than simpler methods like Euler or RK2.

In [ ]:
def rk4_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 4th-order Runge-Kutta method (RK4).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Calculate the RK4 coefficients
        k1 = f(t_n, y_n, alpha)
        k2 = f(t_n + h / 2, y_n + h / 2 * k1, alpha)
        k3 = f(t_n + h / 2, y_n + h / 2 * k2, alpha)
        k4 = f(t_n + h, y_n + h * k3, alpha)

        # Update the solution for the next time step
        y_new = y_n + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        y_values[n + 1] = y_new

    return t_values, y_values

# Adams-Bashforth-Moulton 2nd-Order Predictor-Corrector Method Explanation

### Objective
The **2nd-Order Adams-Bashforth-Moulton Method** is a **predictor-corrector** technique for solving **ordinary differential equations (ODEs)**. This method uses two steps:
1. **Adams-Bashforth (Predictor)**: A two-step explicit formula that predicts the value of the solution at the next step.
2. **Adams-Moulton (Corrector)**: A one-step implicit trapezoidal formula that corrects the prediction using a more accurate estimate.

This method is more accurate than simpler methods like Euler’s method and is useful when higher accuracy is needed while maintaining computational efficiency.

### Key Equations

For a general initial value problem:

$$
\frac{dy}{dt} = f(t, y), \quad y(t_0) = y_0
$$

where $y(t)$ is the function we are solving for, $t$ is the independent variable (usually time), and $f(t, y)$ is the derivative of $y$ with respect to $t$.

The **Adams-Bashforth-Moulton 2nd-order method** uses two formulas:

1. **Predictor (Adams-Bashforth 2nd-order)**:
   $$ 
   y_{n+1}^{(p)} = y_n + \frac{h}{2} \left( 3f(t_n, y_n) - f(t_{n-1}, y_{n-1}) \right)
   $$

2. **Corrector (Adams-Moulton 2nd-order)**:
   $$ 
   y_{n+1} = y_n + \frac{h}{2} \left( f(t_{n+1}, y_{n+1}^{(p)}) + f(t_n, y_n) \right)
   $$

Where:
- $y_{n+1}^{(p)}$ is the predicted value of $y$ at the next time step.
- $h$ is the step size.
- $f(t_n, y_n)$ is the derivative at the current step, and $f(t_{n-1}, y_{n-1})$ is the derivative at the previous step.
- The corrector uses the predicted value $y_{n+1}^{(p)}$ and computes a more accurate value of $y_{n+1}$.

### Step-by-Step Procedure

1. **Initialization**:
   - Start with initial values: $y_0 = y(t_0)$, and an estimate of $y_1$, which can be obtained using another method like Euler’s method or Runge-Kutta.

2. **Prediction (Adams-Bashforth)**:
   - Use the **Adams-Bashforth** formula to predict $y_{n+1}^{(p)}$ based on values at the current and previous steps.

3. **Correction (Adams-Moulton)**:
   - Use the **Adams-Moulton** formula to correct the prediction, giving a more accurate value for $y_{n+1}$.

4. **Iteration**:
   - Repeat the prediction and correction process for each time step until the desired time range is covered.

### Advantages
- **Higher Accuracy**: The method provides more accurate results than lower-order methods like Euler's or the 2nd-order Runge-Kutta method.
- **Efficiency**: It strikes a good balance between accuracy and computational cost, requiring fewer function evaluations than higher-order methods like RK4.

### Disadvantages
- **Requires Two Initial Steps**: To start the method, values for two time steps are required. Typically, the first step is calculated using another method like Euler or Runge-Kutta.
- **Implicit Correction**: The underlying Adams-Moulton formula is implicit, but this implementation substitutes the predicted value and corrects once (PECE).

### Summary
The **2nd-Order Adams-Bashforth-Moulton Method** is a predictor-corrector method that improves both accuracy and stability over simple methods like Euler’s method. It uses a combination of explicit prediction and implicit correction, making it a practical choice for many applications requiring a balance between accuracy and computational efficiency.

## Explanation of the `adams_bashforth_moulton_2nd` Function

This function implements the **2nd-order Adams-Bashforth-Moulton predictor-corrector method** for solving ordinary differential equations (ODEs). The method combines two steps: the **Adams-Bashforth method** for the prediction and the **Adams-Moulton method** for the correction. This approach is more accurate than explicit methods and leverages a two-step process to improve the solution.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **First Step using RK4 Method**:
   - Since the Adams-Bashforth-Moulton method requires two initial points, the **RK4 method** is used to compute the second point. The first two points in `y_values` are set using RK4.

3. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are used.
   - The predictor-corrector process is applied in two steps:
   
   **Predictor Step (Adams-Bashforth 2nd order)**:
   - The predictor step uses the current and previous function values \( f_n \) and \( f_{n-1} \) to estimate the next solution \( y_{\text{predictor}} \):
     \[
     y_{\text{predictor}} = y_n + \frac{h}{2} \cdot (3 \cdot f_n - f_{n-1})
     \]
     This step predicts the solution at the next time point \( t_{n+1} \).

   **Corrector Step (Adams-Moulton 2nd order)**:
   - The corrector step refines the predicted value by using the function values at the predicted point \( f_{\text{predictor}} \) and the current function value \( f_n \):
     \[
     y_{\text{corrector}} = y_n + \frac{h}{2} \cdot (f_{\text{predictor}} + f_n)
     \]
     This corrected value improves the accuracy of the solution at \( t_{n+1} \).

4. **Update the Solution**:
   - The corrected value \( y_{\text{corrector}} \) is stored in `y_values[n+1]`.

5. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **First Step (RK4)**: Since the Adams-Bashforth-Moulton method requires two initial points, the first step is solved using RK4.
2. **Predictor Step**: The Adams-Bashforth method predicts the next value using a combination of current and previous slopes.
3. **Corrector Step**: The Adams-Moulton method refines the prediction by correcting the solution with the average slope at the predicted and current points.
4. **Improved Accuracy**: By applying both prediction and correction, the method offers improved accuracy compared to simpler methods like Euler.

### Key Advantages:
- **Improved Accuracy**: The predictor-corrector method is more accurate than basic explicit methods because it refines the predicted solution using a correction step.
- **Multi-Step Method**: The use of two points in the Adams-Bashforth step makes it more stable and efficient for solving ODEs.

### Limitations:
- **Requires Two Initial Points**: The Adams-Bashforth-Moulton method requires at least two points to begin, which is why an initial method like RK4 is used for the first step.
- **More Computational Effort**: This method involves both prediction and correction, requiring multiple function evaluations per step.

In [ ]:
def adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 2nd-order Adams-Bashforth-Moulton predictor-corrector method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    # Start with RK4 to get the second point (since Adams methods require two initial points)
    t_values_rk4, y_values_rk4 = rk4_method(f, y0, t0, t0 + h, h, alpha)
    y_values[1] = y_values_rk4[-1]

    for n in range(1, N):
        t_n = t_values[n]
        t_np1 = t_values[n + 1]  # t_n+1
        y_n = y_values[n]
        y_nm1 = y_values[n - 1]

        # Predictor step (Adams-Bashforth 2nd order)
        f_n = f(t_n, y_n, alpha)
        f_nm1 = f(t_n - h, y_nm1, alpha)
        y_predictor = y_n + (h / 2) * (3 * f_n - f_nm1)

        # Corrector step (Adams-Moulton 2nd order)
        f_predictor = f(t_np1, y_predictor, alpha)
        y_corrector = y_n + (h / 2) * (f_predictor + f_n)

        # Update the solution
        y_values[n + 1] = y_corrector

    return t_values, y_values

In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [2]  # Different alpha values for comparison
    
    # Prepare for plotting (two subplots: left for error, right for time)
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors and computation times for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        rk4_errors = []
        adams_errors = []
        euler_times = []
        heun_times = []
        midpoint_times = []
        rk4_times = []
        adams_times = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15} | {'RK4 Error':>15} | {'Adams Error':>15}")
        print("-" * 105)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            start_time = time.perf_counter()
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            euler_time = end_time - start_time
            euler_times.append(euler_time)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            start_time = time.perf_counter()
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            heun_time = end_time - start_time
            heun_times.append(heun_time)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            start_time = time.perf_counter()
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            midpoint_time = end_time - start_time
            midpoint_times.append(midpoint_time)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # RK4 Method
            start_time = time.perf_counter()
            _, y_values_rk4 = rk4_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            rk4_time = end_time - start_time
            rk4_times.append(rk4_time)
            y_rk4 = y_values_rk4[-1]
            rk4_error = abs(y_rk4 - y_true) / abs(y_true)
            rk4_errors.append(rk4_error)

            # Adams-Bashforth-Moulton Method
            start_time = time.perf_counter()
            _, y_values_adams = adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams_time = end_time - start_time
            adams_times.append(adams_time)
            y_adams = y_values_adams[-1]
            adams_error = abs(y_adams - y_true) / abs(y_true)
            adams_errors.append(adams_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e} | {rk4_error:15.10e} | {adams_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale (left plot)
        ax1.loglog(step_sizes, euler_errors, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax1.loglog(step_sizes, heun_errors, 'x--', markersize=10, lw=4, label=f'Heun alpha={alpha}')
        ax1.loglog(step_sizes, midpoint_errors, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
        ax1.loglog(step_sizes, rk4_errors, 'd-', markersize=10, lw=4, label=f'RK4 alpha={alpha}')
        ax1.loglog(step_sizes, adams_errors, 'p-', markersize=10, lw=4, label=f'Adams alpha={alpha}')
        
        # Plot computation times for each method on a log-log scale (right plot)
        ax2.loglog(step_sizes, euler_times, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax2.loglog(step_sizes, heun_times, 'x--', markersize=10, lw=4, label=f'Heun alpha={alpha}')
        ax2.loglog(step_sizes, midpoint_times, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
        ax2.loglog(step_sizes, rk4_times, 'd-', markersize=10, lw=4, label=f'RK4 alpha={alpha}')
        ax2.loglog(step_sizes, adams_times, 'p-', markersize=10, lw=4, label=f'Adams alpha={alpha}')
    
    # Set labels and titles for the left plot (relative error)
    ax1.set_xlabel('Step Size h')
    ax1.set_ylabel('Relative Error')
    ax1.set_title('Relative Error vs. Step size')
    ax1.grid(True, which="both", ls="--")
    
    # Set labels and titles for the right plot (computation time)
    ax2.set_xlabel('Step Size h')
    ax2.set_ylabel('Computation Time (s)')
    ax2.set_title('Computation Time vs. Step size')
    ax2.grid(True, which="both", ls="--")
    
    # Add legends
    ax1.legend()
    ax2.legend()
    
    # Display the plots
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    main()

# Adams-Bashforth-Moulton 4th-Order Predictor-Corrector Method Explanation

### Objective
The **4th-Order Adams-Bashforth-Moulton Method** is a higher-order **predictor-corrector** technique used for solving **ordinary differential equations (ODEs)**. It combines the **Adams-Bashforth** explicit method (predictor) and the **Adams-Moulton** implicit method (corrector) to achieve higher accuracy. This method is particularly useful for systems that require precise long-term integration.

### Key Equations

For a general initial value problem:

$$
\frac{dy}{dt} = f(t, y), \quad y(t_0) = y_0
$$

where $y(t)$ is the function we are solving for, $t$ is the independent variable (usually time), and $f(t, y)$ is the derivative of $y$ with respect to $t$.

The **4th-Order Adams-Bashforth-Moulton method** consists of two main steps:

1. **Predictor (Adams-Bashforth 4th-order)**:
   $$ 
   y_{n+1}^{(p)} = y_n + \frac{h}{24} \left( 55f(t_n, y_n) - 59f(t_{n-1}, y_{n-1}) + 37f(t_{n-2}, y_{n-2}) - 9f(t_{n-3}, y_{n-3}) \right)
   $$

2. **Corrector (Adams-Moulton 4th-order)**:
   $$ 
   y_{n+1} = y_n + \frac{h}{24} \left( 9f(t_{n+1}, y_{n+1}^{(p)}) + 19f(t_n, y_n) - 5f(t_{n-1}, y_{n-1}) + f(t_{n-2}, y_{n-2}) \right)
   $$

Where:
- $y_{n+1}^{(p)}$ is the predicted value of $y$ at the next step.
- $h$ is the time step size.
- $f(t_n, y_n)$ is the derivative at the current step, $f(t_{n-1}, y_{n-1})$, $f(t_{n-2}, y_{n-2})$, and $f(t_{n-3}, y_{n-3})$ are derivatives from previous steps.
- The corrector improves upon the predicted value $y_{n+1}^{(p)}$, yielding a more accurate result for $y_{n+1}$.

### Step-by-Step Procedure

1. **Initialization**:
   - Start with the initial value $y_0 = y(t_0)$ and compute the first few steps using a method like the **4th-order Runge-Kutta (RK4)**, as the predictor-corrector method needs values from previous steps.
   - Compute $y_1, y_2, y_3$ using RK4 or another method.

2. **Prediction (Adams-Bashforth)**:
   - Use the **Adams-Bashforth** formula to predict $y_{n+1}^{(p)}$ based on the values at the current and previous steps.

3. **Correction (Adams-Moulton)**:
   - Use the **Adams-Moulton** formula to correct the predicted value $y_{n+1}^{(p)}$, giving a more accurate value for $y_{n+1}$.

4. **Iteration**:
   - Repeat the prediction and correction process for each subsequent time step until the end of the desired time range.

### Advantages
- **Higher Accuracy**: The method provides fourth-order accuracy, making it significantly more precise than lower-order methods like Euler’s method or the 2nd-order Runge-Kutta method.
- **Efficiency**: The predictor-corrector approach balances accuracy with computational efficiency, making it useful for long-term simulations.

### Disadvantages
- **Complexity**: The method requires multiple past values (four steps) to calculate each new value, which can make implementation more complex.
- **Implicit Correction**: The implementation evaluates the corrector once at the predicted state (PECE); it does not solve the implicit equation to convergence.

### Summary
The **4th-Order Adams-Bashforth-Moulton Method** is a highly accurate predictor-corrector method that combines explicit and implicit steps to achieve fourth-order accuracy. It is well-suited for applications where both accuracy and efficiency are needed, but its complexity and need for multiple initial values may make it less suitable for simpler problems.

## Explanation of the `adams_bashforth_moulton_4th` Function

This function implements the **4th-order Adams-Bashforth-Moulton predictor-corrector method** for solving ordinary differential equations (ODEs). The method combines the **Adams-Bashforth 4th-order method** for predicting the next solution point and the **Adams-Moulton 4th-order method** for correcting the prediction, resulting in high accuracy for each time step.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **First Three Steps using RK4 Method**:
   - Since the Adams-Bashforth-Moulton 4th-order method requires four initial points, the **RK4 method** is used to compute the first four points. This ensures that the method has enough prior values to perform the predictor-corrector steps.

3. **Main Loop for Time Stepping**:
   - For each time step \( n \), starting from the fourth step \( n = 3 \), the current time \( t_n \) and the previous solution points \( y_n, y_{n-1}, y_{n-2}, y_{n-3} \) are used to predict and correct the solution.

4. **Predictor Step (Adams-Bashforth 4th order)**:
   - The Adams-Bashforth 4th-order method uses the current and previous function evaluations to predict the next solution point \( y_{\text{predictor}} \):
     \[
     y_{\text{predictor}} = y_n + \frac{h}{24} \cdot (55 f_n - 59 f_{n-1} + 37 f_{n-2} - 9 f_{n-3})
     \]
   - This step predicts the value of \( y \) at the next time point \( t_{n+1} \).

5. **Corrector Step (Adams-Moulton 4th order)**:
   - The Adams-Moulton 4th-order method corrects the predicted value by using a combination of the function evaluation at the predicted point and previous function evaluations:
     \[
     y_{\text{corrector}} = y_n + \frac{h}{24} \cdot (9 f_{\text{predictor}} + 19 f_n - 5 f_{n-1} + f_{n-2})
     \]
   - This step refines the predicted value, resulting in a more accurate solution at \( t_{n+1} \).

6. **Update the Solution**:
   - The corrected value \( y_{\text{corrector}} \) is stored in `y_values[n+1]`.

7. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **First Three Steps (RK4)**: The method starts by using RK4 for the first four points to initialize the Adams-Bashforth-Moulton process.
2. **Predictor Step**: The 4th-order Adams-Bashforth method predicts the next value based on the current and three previous slopes.
3. **Corrector Step**: The 4th-order Adams-Moulton method refines the predicted value by combining the predicted and current slopes.
4. **Higher Accuracy**: The use of both prediction and correction steps provides fourth-order accuracy, significantly reducing the error at each step.

### Key Advantages:
- **High Accuracy**: The combination of Adams-Bashforth and Adams-Moulton methods gives this approach fourth-order accuracy, meaning the error decreases as \( O(h^4) \).
- **Predictor-Corrector Approach**: By using both a prediction and correction step, this method refines the solution at each time step, improving stability and accuracy.

### Limitations:
- **Requires Four Initial Points**: The 4th-order method requires at least four points to begin, which is why the RK4 method is used for the first three steps.
- **More Computational Effort**: This method involves multiple function evaluations per step, making it computationally more expensive than lower-order methods.

In [ ]:
def adams_bashforth_moulton_4th(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 4th-order Adams-Bashforth-Moulton predictor-corrector method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    # Use RK4 to initialize the first four points
    for i in range(min(3, N)):
        t_values_rk4, y_values_rk4 = rk4_method(f, y_values[i], t0 + i * h, t0 + (i + 1) * h, h, alpha)
        y_values[i + 1] = y_values_rk4[-1]

    for n in range(3, N):
        t_n = t_values[n]
        t_np1 = t_values[n + 1]  # t_n+1
        y_n = y_values[n]
        y_nm1 = y_values[n - 1]
        y_nm2 = y_values[n - 2]
        y_nm3 = y_values[n - 3]

        # Predictor step (Adams-Bashforth 4th order)
        f_n = f(t_n, y_n, alpha)
        f_nm1 = f(t_n - h, y_nm1, alpha)
        f_nm2 = f(t_n - 2 * h, y_nm2, alpha)
        f_nm3 = f(t_n - 3 * h, y_nm3, alpha)
        y_predictor = y_n + (h / 24) * (55 * f_n - 59 * f_nm1 + 37 * f_nm2 - 9 * f_nm3)

        # Corrector step (Adams-Moulton 4th order)
        f_predictor = f(t_np1, y_predictor, alpha)
        y_corrector = y_n + (h / 24) * (9 * f_predictor + 19 * f_n - 5 * f_nm1 + f_nm2)

        # Update the solution
        y_values[n + 1] = y_corrector

    return t_values, y_values

In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [2]  # Different alpha values for comparison
    
    # Prepare for plotting (two subplots: left for error, right for time)
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors and computation times for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        rk4_errors = []
        adams2_errors = []
        adams4_errors = []
        euler_times = []
        heun_times = []
        midpoint_times = []
        rk4_times = []
        adams2_times = []
        adams4_times = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15} | {'RK4 Error':>15} | {'Adams2 Error':>15} | {'Adams4 Error':>15}")
        print("-" * 120)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            start_time = time.perf_counter()
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            euler_time = end_time - start_time
            euler_times.append(euler_time)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            start_time = time.perf_counter()
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            heun_time = end_time - start_time
            heun_times.append(heun_time)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            start_time = time.perf_counter()
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            midpoint_time = end_time - start_time
            midpoint_times.append(midpoint_time)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # RK4 Method
            start_time = time.perf_counter()
            _, y_values_rk4 = rk4_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            rk4_time = end_time - start_time
            rk4_times.append(rk4_time)
            y_rk4 = y_values_rk4[-1]
            rk4_error = abs(y_rk4 - y_true) / abs(y_true)
            rk4_errors.append(rk4_error)

            # Adams-Bashforth-Moulton 2nd Order Method
            start_time = time.perf_counter()
            _, y_values_adams2 = adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams2_time = end_time - start_time
            adams2_times.append(adams2_time)
            y_adams2 = y_values_adams2[-1]
            adams2_error = abs(y_adams2 - y_true) / abs(y_true)
            adams2_errors.append(adams2_error)

            # Adams-Bashforth-Moulton 4th Order Method
            start_time = time.perf_counter()
            _, y_values_adams4 = adams_bashforth_moulton_4th(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams4_time = end_time - start_time
            adams4_times.append(adams4_time)
            y_adams4 = y_values_adams4[-1]
            adams4_error = abs(y_adams4 - y_true) / abs(y_true)
            adams4_errors.append(adams4_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e} | {rk4_error:15.10e} | {adams2_error:15.10e} | {adams4_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale (left plot)
        ax1.loglog(step_sizes, euler_errors, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax1.loglog(step_sizes, heun_errors, 'x--', markersize=10, lw=4, label=f'Heun alpha={alpha}')
        ax1.loglog(step_sizes, midpoint_errors, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
        ax1.loglog(step_sizes, rk4_errors, 'd-', markersize=10, lw=4, label=f'RK4 alpha={alpha}')
        ax1.loglog(step_sizes, adams2_errors, 'p-', markersize=10, lw=4, label=f'Adams2 alpha={alpha}')
        ax1.loglog(step_sizes, adams4_errors, '*-', markersize=10, lw=4, label=f'Adams4 alpha={alpha}')
        
        # Plot computation times for each method on a log-log scale (right plot)
        ax2.loglog(step_sizes, euler_times, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax2.loglog(step_sizes, heun_times, 'x--', markersize=10, lw=4, label=f'Heun alpha={alpha}')
        ax2.loglog(step_sizes, midpoint_times, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
        ax2.loglog(step_sizes, rk4_times, 'd-', markersize=10, lw=4, label=f'RK4 alpha={alpha}')
        ax2.loglog(step_sizes, adams2_times, 'p-', markersize=10, lw=4, label=f'Adams2 alpha={alpha}')
        ax2.loglog(step_sizes, adams4_times, '*-', markersize=10, lw=4, label=f'Adams4 alpha={alpha}')
    
    # Set labels and titles for the left plot (relative error)
    ax1.set_xlabel('Step Size h')
    ax1.set_ylabel('Relative Error')
    ax1.set_title('Relative Error vs. Step size')
    ax1.grid(True, which="both", ls="--")
    
    # Set labels and titles for the right plot (computation time)
    ax2.set_xlabel('Step Size h')
    ax2.set_ylabel('Computation Time (s)')
    ax2.set_title('Computation Time vs. Step size')
    ax2.grid(True, which="both", ls="--")
    
    # Add legends
    ax1.legend()
    ax2.legend()
    
    # Display the plots
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()